# Monitoring, Drift, and MLOps

Drift metrics, W&B artifact posture, and reproducibility checks.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.chdir(ROOT)
SAMPLE_ROWS = int(os.getenv("GTD_NOTEBOOK_SAMPLE_ROWS", "25000"))
print(f"Project root: {ROOT}")
print(f"Notebook sample rows: {SAMPLE_ROWS:,}")

Project root: C:\Users\kunmi\Personal Projects\Global-Terrorism-Database
Notebook sample rows: 25,000


In [2]:
import pandas as pd

from gtd_capstone.data.repository import DataRepository
from gtd_capstone.monitoring.drift import drift_report

incidents = DataRepository().load_incidents(sample_rows=SAMPLE_ROWS)
report = drift_report(
    incidents,
    split_year=2014,
    numeric_columns=["nkill", "nwound", "casualties", "severity_score"],
    categorical_columns=["region_txt", "attacktype1_txt", "severity"],
)
{"severity": report["severity"], "max_score": report["max_score"]}

{'severity': 'high', 'max_score': 0.34657359027997264}

In [3]:
rows = [
    {"metric": key, "score": value, "kind": "numeric_psi"}
    for key, value in report["numeric_psi"].items()
] + [
    {"metric": key, "score": value, "kind": "categorical_js"}
    for key, value in report["categorical_js"].items()
]
pd.DataFrame(rows).sort_values("score", ascending=False)

,metric,score,kind
6,severity,0.346574,categorical_js
5,attacktype1_txt,0.346574,categorical_js
4,region_txt,0.346574,categorical_js
2,casualties,0.000000,numeric_psi
1,nwound,0.000000,numeric_psi
0,nkill,0.000000,numeric_psi
3,severity_score,0.000000,numeric_psi


In [4]:
model_dir = ROOT / "artifacts" / "models"
wandb_dir = ROOT / "wandb"
{
    "model_artifacts": sorted(path.name for path in model_dir.glob("*"))[:10]
    if model_dir.exists()
    else [],
    "wandb_offline_runs": len(list(wandb_dir.glob("offline-run-*")))
    if wandb_dir.exists()
    else 0,
    "notebook_sample_rows": SAMPLE_ROWS,
}

{'model_artifacts': ['severity_classifier.joblib',
  'severity_extra_trees.joblib',
  'severity_metrics.json'],
 'wandb_offline_runs': 21,
 'notebook_sample_rows': 25000}